# Experimental Benchmark: Neural Networks and Fuzzy / Neuro-Fuzzy Models

This notebook evaluates artificial neural networks and fuzzy/neuro-fuzzy systems on four real classification datasets. The focus is the experimental methodology: dataset description, exploratory analysis, controlled train/validation/test splits, hyperparameter selection, repeated runs, benchmark tables, confusion matrices and statistical comparison.

## 1. Environment setup

The notebook only orchestrates the experiment. Implementations are kept under `src/neural_fuzzy_benchmark/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "neural_fuzzy_benchmark").exists():
            return candidate
    raise FileNotFoundError("Open the notebook from the extracted project root.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "raw"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from neural_fuzzy_benchmark.data.class_grouping import ClassGroupingPolicy, audit_class_distribution
from neural_fuzzy_benchmark.data.loaders import DatasetLoader
from neural_fuzzy_benchmark.data.tables import (
    build_class_distribution_table,
    build_data_dictionary_table,
    build_dataset_metadata_table,
    build_feature_summary_table,
)
from neural_fuzzy_benchmark.evaluation.statistical_tests import (
    friedman_tests_by_dataset,
    global_friedman_test,
    global_pairwise_wilcoxon_tests,
    pairwise_wilcoxon_tests_by_dataset,
)
from neural_fuzzy_benchmark.experiments.runner import ExperimentSettings, run_experiment_suite, summarize_metrics
from neural_fuzzy_benchmark.reporting.benchmark_tables import (
    build_benchmark_table,
    build_best_parameter_frequency,
    build_metric_matrix,
    build_model_comparison_table,
    rank_algorithms_by_dataset,
)
from neural_fuzzy_benchmark.visualization.plots import (
    plot_confusion_matrix_from_results,
    plot_metric_by_dataset,
    plot_metric_heatmap,
)

In [ ]:
%load_ext watermark
%watermark -v -m -p numpy,pandas,scipy,matplotlib,watermark

## 2. Experimental settings

The protocol uses 60% training, 20% validation and 20% final testing. Hyperparameters are selected on validation data. The test split is only used after model selection. The 21 runs use different seeds while preserving reproducibility.

In [ ]:
N_RUNS = 21
BASE_SEED = 2026016

class_grouping_policy = ClassGroupingPolicy(
    enabled=True,
    max_classes=3,
    min_class_share=0.05,
    min_class_count=15,
)

settings = ExperimentSettings(
    n_runs=N_RUNS,
    base_seed=BASE_SEED,
    train_fraction=0.60,
    validation_fraction=0.20,
    class_grouping_policy=class_grouping_policy,
)

## 3. Dataset loading

The selected datasets are real tabular classification problems with continuous numerical attributes. This is suitable for fuzzy and neuro-fuzzy models because Gaussian membership functions and RBF antecedents are naturally defined over distances in continuous feature spaces.

In [ ]:
loader = DatasetLoader(data_dir=DATA_DIR)
datasets = loader.load_all()

list(datasets.keys())

## 4. Dataset origin and problem description

Each dataset is documented with source, task type, target variable, number of samples, number of attributes and class distribution.

In [ ]:
dataset_metadata = build_dataset_metadata_table(datasets)
display(dataset_metadata)

## 5. Data dictionary

The table below documents the target and input variables. The fuzzy models are especially appropriate here because the attributes are continuous measurements such as lengths, areas, perimeters, chemical concentrations and geometric descriptors.

In [ ]:
data_dictionary = build_data_dictionary_table()
display(data_dictionary)

## 6. Exploratory analysis

The exploratory analysis summarizes class balance and the numerical distribution of the input variables.

In [ ]:
class_distribution = build_class_distribution_table(datasets)
display(class_distribution)

In [ ]:
feature_summary = build_feature_summary_table(datasets)
display(feature_summary)

## 7. Class grouping audit

The class grouping policy is only applied when a dataset has too many classes or low-represented classes. Wine already has three classes, so no artificial aggregation is required under this policy.

In [ ]:
for dataset in datasets.values():
    print(dataset.title)
    display(audit_class_distribution(dataset.target, dataset.class_names, class_grouping_policy))

## 8. Model comparison table

The four implemented models satisfy the required comparison between neural networks and fuzzy/neuro-fuzzy approaches.

In [ ]:
model_comparison = build_model_comparison_table()
display(model_comparison)

## 9. Experimental execution

For each dataset and each run, the same train/validation/test split is used across all algorithms. Each algorithm evaluates a hyperparameter grid, selects the best configuration by validation macro F1, refits on training plus validation data and evaluates on the held-out test set.

In [ ]:
results = run_experiment_suite(datasets, settings)
display(results.head())

## 10. Benchmark tables

The benchmark is reported as mean ± standard deviation across the independent runs. Macro F1 is used as the main comparison metric because it gives equal importance to each class.

In [ ]:
summary = summarize_metrics(results)
benchmark_table = build_benchmark_table(summary)
metric_matrix = build_metric_matrix(summary, metric="f1_macro")
rankings = rank_algorithms_by_dataset(summary, metric="f1_macro")

display(benchmark_table)
display(metric_matrix)
display(rankings)

## 11. Selected hyperparameters

The table below shows how often each configuration was selected as the best validation configuration across runs.

In [ ]:
best_parameter_frequency = build_best_parameter_frequency(results)
display(best_parameter_frequency)

## 12. Statistical comparison

The Friedman test compares all algorithms simultaneously. Pairwise Wilcoxon tests are then used as post-hoc comparisons, with Bonferroni and Holm corrections for multiple pairwise tests.

In [ ]:
friedman_f1_by_dataset = friedman_tests_by_dataset(results, metric="f1_macro")
global_friedman_f1 = global_friedman_test(results, metric="f1_macro")

display(friedman_f1_by_dataset)
display(global_friedman_f1)

In [ ]:
pairwise_f1_by_dataset = pairwise_wilcoxon_tests_by_dataset(results, metric="f1_macro")
global_pairwise_f1 = global_pairwise_wilcoxon_tests(results, metric="f1_macro")

display(pairwise_f1_by_dataset)
display(global_pairwise_f1)

## 13. Visual benchmark

The plots summarize the average macro F1 by dataset and algorithm.

In [ ]:
plot_metric_by_dataset(summary, metric="f1_macro")

In [ ]:
plot_metric_heatmap(summary, metric="f1_macro")

## 14. Confusion matrix

The confusion matrix uses a light blue gradient. Change the dataset, algorithm and run to inspect other combinations.

In [ ]:
plot_confusion_matrix_from_results(
    results,
    dataset="wine",
    algorithm="Gaussian_Fuzzy_Rules",
    run=1,
)

## 15. Technical interpretation guide

The final discussion should compare performance, stability, hyperparameter sensitivity, computational cost and interpretability. The fuzzy models should be discussed in relation to the continuous nature of the variables: Gaussian rules and RBF antecedents are meaningful because the datasets are based on numerical measurements where distance and proximity have practical interpretation.